# 08. Обучение baseline на corrected_v1

Обучает ту же RuBERT Token Classification модель на трёх seed без изменения гиперпараметров. Это изолирует влияние разметки от изменения архитектуры. Каждый запуск сохраняется отдельно.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
from pathlib import Path
import runpy
PROJECT_DIR = Path('/content/drive/MyDrive/NER_RuREBus_project')
runpy.run_path(str(PROJECT_DIR / 'colab_bootstrap.py'))['bootstrap_project'](PROJECT_DIR)
MANIFEST = PROJECT_DIR / 'rurebus_data/versions/corrected_v1/manifest.csv'
if not MANIFEST.is_file():
    raise FileNotFoundError('Сначала выполните 07_build_corrected_v1.ipynb')
CONFIGS = [
    PROJECT_DIR / 'configs/experiments/ner_baseline_corrected_v1.yaml',
    PROJECT_DIR / 'configs/experiments/ner_baseline_corrected_v1_seed17.yaml',
    PROJECT_DIR / 'configs/experiments/ner_baseline_corrected_v1_seed73.yaml',
]
ORIGINAL_DATA_CONFIG = PROJECT_DIR / 'configs/data/rurebus.yaml'

In [ ]:
import torch
print('PyTorch:', torch.__version__)
print('CUDA:', torch.cuda.is_available())
if not torch.cuda.is_available():
    raise RuntimeError('Для трёх полноценных запусков рекомендуется GPU runtime.')
print('GPU:', torch.cuda.get_device_name(0))

In [ ]:
from rurebus_ie.training import train_ner_experiment, test_ner_experiment, evaluate_ner_experiment, register_ner_baseline
runs = []
for config in CONFIGS:
    print('\n===', config.name, '===')
    summary = train_ner_experiment(config, project_root=PROJECT_DIR)
    test_result = test_ner_experiment(config, project_root=PROJECT_DIR)
    official_test_result = evaluate_ner_experiment(
        config, split_key='test', project_root=PROJECT_DIR,
        evaluation_data_config_path=ORIGINAL_DATA_CONFIG,
        artifact_prefix='official_original_test',
    )
    alias = f"B1-corrected-seed{config.stem.split('seed')[-1] if 'seed' in config.stem else '42'}"
    record = register_ner_baseline(config, alias=alias, project_root=PROJECT_DIR)
    runs.append({
        'config': config.name,
        'best_epoch': summary.best_epoch,
        'validation_micro_f1': summary.best_validation_f1,
        'test_micro_f1': test_result.metrics.micro_f1,
        'test_macro_f1': test_result.metrics.macro_f1,
        'official_original_test_micro_f1': official_test_result.metrics.micro_f1,
        'alias': record['alias'],
    })

In [ ]:
import pandas as pd
runs_df = pd.DataFrame(runs)
display(runs_df)
display(runs_df[['validation_micro_f1','test_micro_f1','test_macro_f1','official_original_test_micro_f1']].agg(['mean','std']))
print('Для сравнения с B0 используйте official_original_test_micro_f1. Обычный test_micro_f1 использует corrected gold.')